In [ ]:
pip install pandas numpy pywavelets matplotlib
!pip install --user scikit-learn
!pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for pandas from https://files.pythonhosted.org/packages/29/d4/1244ab8edf173a10fd601f7e13b9566c1b525c4f365d6bee918e68381889/pandas-2.2.3-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for numpy from https://files.pythonhosted.org/packages/42/6e/55580a538116d16ae7c9aa17d4edd56e83f42126cb1dfe7a684da7925d2c/numpy-2.2.3-cp312-cp312-win_amd64.whl.metadata
     ---------------------------------------- 0.0/60.8 kB ? eta -:--:--
     ------ --------------------------------- 10.2/60.8 kB ? eta -:--:--
     ------------------- ------------------ 30.7/60.8 kB 325.1 kB/s eta 0:00:01
     ------------------------- ------------ 41.0/60.8 kB 326.8 kB/s eta 0:00:01
     -------------------------------------- 60.8/60.8 kB 404.8 kB/s eta 0:00:00
  Obtaining dependency information for pywavelets from https://files.pythonhosted.org/packages/1c/88/9e2aa9d5fde08bfc0fb18ffb1b


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import pandas as pd
import numpy as np
import pywt
import matplotlib.pyplot as plt
import math

In [10]:
from sklearn.preprocessing import StandardScaler

In [21]:
e1 = pd.read_excel(r"c:\Users\ma.benavidess1\Downloads\EICH101.xlsx")

# Filtrar registros según ORIG_RAW_VOLUME
e1 = e1[e1["ORIG_RAW_VOLUME"] <= 60]

# Convertir EFFECTIVE_DATE a formato de fecha
e1["EFFECTIVE_DATE"] = pd.to_datetime(e1["EFFECTIVE_DATE"], errors='coerce')

# Aplicar normalización Z-score
scaler = StandardScaler()
columns_to_normalize = ["STD_VOLUME", "ORIG_STD_VOLUME", "ORIG_TEMPERATURE","TEMPERATURE","PRESSURE","ORIG_PRESSURE","RAW_VOLUME","ORIG_RAW_VOLUME"]  # Ajusta si necesitas más
e1[columns_to_normalize] = scaler.fit_transform(e1[columns_to_normalize])

# Guardar el resultado en un CSV en la misma carpeta donde está el pipeline
output_path = "datos_procesados_empresa1.csv"
e1.to_csv(output_path, index=False)

print(f"Archivo guardado en {output_path}")

Archivo guardado en datos_procesados_empresa1.csv


In [23]:
# Calcular el percentil 99 global para evitar que las anomalías dominen
global_max = np.percentile(e1["ORIG_RAW_VOLUME"], 99)
 
# Aplicar normalización dividiendo por ese valor máximo
e1["ORIG_RAW_VOLUME_NORM"] = e1["ORIG_RAW_VOLUME"] / global_max
e1["ORIG_RAW_VOLUME_NORM"] = np.clip(e1["ORIG_RAW_VOLUME_NORM"], 0, 1)

In [24]:
# Crear la carpeta donde se guardarán las imágenes
output_folder = r"c:\Users\ma.benavidess1\Downloads\ProyectoGradoGEB\imagenes_diarias"
os.makedirs(output_folder, exist_ok=True)


In [26]:
# Escalas para la Transformada Wavelet Continua (CWT)
scales = np.arange(1, 128)

# Recorrer el archivo secuencialmente
e1_sorted = e1.sort_values(by="EFFECTIVE_DATE")
 
# Obtener los años únicos en el DataFrame
years = e1_sorted['EFFECTIVE_DATE'].dt.year.unique()
 
for year in years:
    e1_year = e1_sorted[e1_sorted['EFFECTIVE_DATE'].dt.year == year]
    
    # Obtener los días únicos ordenados
    days = sorted(e1_year['EFFECTIVE_DATE'].dt.date.unique())
 
    for day in days:
        e1_day = e1_year[e1_year['EFFECTIVE_DATE'].dt.date == day]
        if not e1_day.empty:
            # Extraer la señal del día (manteniendo valores originales sin escalar)
            signal_day = e1_day['ORIG_RAW_VOLUME'].values
 
            # Aplicar la Transformada Wavelet Continua (CWT) con Morlet
            coefficients, frequencies = pywt.cwt(signal_day, scales, 'morl')
 
            # Crear la figura del escalograma
            plt.figure(figsize=(10, 6))
            plt.imshow(np.abs(coefficients), aspect="auto", cmap="jet",
                       extent=[0, len(signal_day), scales[-1], scales[0]])

            plt.colorbar(label="Magnitud")
            plt.title(f"Escalograma - Año {year} - Día {day}", fontsize=12, fontweight="bold")
            plt.xlabel("Horas del Día")
            plt.ylabel("Escala")
 
            # Guardar la imagen
            filename = f"Año{year}_Día{day}.png".replace(" ", "")
            filepath = os.path.join(output_folder, filename)
            plt.savefig(filepath, dpi=300, bbox_inches="tight")
            plt.close()
 
# Listar los archivos generados
generated_files = os.listdir(output_folder)

print("Ejemplo de archivos generados:", generated_files[:10])  # Mostrar solo los primeros 10 archivos como muestra

Ejemplo de archivos generados: ['Año2018_Día2018-09-03.png', 'Año2018_Día2018-09-04.png', 'Año2018_Día2018-09-05.png', 'Año2018_Día2018-09-06.png', 'Año2018_Día2018-09-07.png', 'Año2018_Día2018-09-08.png', 'Año2018_Día2018-09-09.png', 'Año2018_Día2018-09-10.png', 'Año2018_Día2018-09-11.png', 'Año2018_Día2018-09-12.png']
